# Author diarization parameter tuning

Use this notebook to try different window settings, stride values, similarity thresholds, and text limits without editing the script each time.

Run the cells in order. Then adjust the parameters in the tuning cell and re-run it.

In [1]:
from __future__ import annotations

from author_prediction.deep_stylometry_encoder import DeepStylometryEncoder
from author_prediction.pipeline_implementation import run_pipeline
from author_prediction.profile_tracker import AuthorProfileTracker
from author_prediction.reporting import format_run_summary, summarize_run
from author_prediction.segmenter import split_into_sentences

DATA_PATH = "src/author_prediction/data/data_ortho.txt"

In [5]:
def load_limited_text(path: str, max_chars: int | None = None, max_sentences: int | None = None) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    if max_chars is not None and max_chars > 0:
        text = text[:max_chars]

    if max_sentences is not None and max_sentences > 0:
        sentences = split_into_sentences(text)
        text = " ".join(sentences[:max_sentences])

    return text


def print_progress(progress: float, processed: int, total: int) -> None:
    pct = int(progress * 100)
    print(f"Processing text: {pct}% ({processed}/{total} sentences)", end="\r")

In [ ]:
# Tune these values and re-run the experiment cell.
MAX_CHARS = 100000
MAX_SENTENCES = None

context_window_size = 20
stride = 5
sim_threshold = 0.94
ema_alpha = 0.22
min_tokens_for_update = 15
merge_threshold = 0.95

text = load_limited_text(DATA_PATH, max_chars=MAX_CHARS, max_sentences=MAX_SENTENCES)
sentences = split_into_sentences(text)
print(f"Loaded {len(sentences)} sentences from {DATA_PATH}")
print(f"Using {len(text)} characters of input text")

In [ ]:
tracker = AuthorProfileTracker(
    sim_threshold=sim_threshold,
    ema_alpha=ema_alpha,
    min_tokens_for_update=min_tokens_for_update,
)
encoder = DeepStylometryEncoder()

result = run_pipeline(
    text,
    encoder=encoder,
    tracker=tracker,
    context_window_size=context_window_size,
    stride=stride,
    merge_threshold=merge_threshold,
    progress_callback=print_progress,
)

print("\n")
summary = summarize_run(result)
print(format_run_summary(summary))

if result.get("merge_events"):
    print("\nAuthor merge events:")
    for event in result["merge_events"]:
        print(
            f"  - Merged {event['merged_from']} into {event['kept']} (similarity={event['similarity']:.4f})"
        )

print("\nPer-sentence detail:")
assignments = result["assignments"]
for i, r in enumerate(assignments):
    changed = i > 0 and assignments[i]["author_id"] != assignments[i - 1]["author_id"]
    sim_str = f"{r['similarity']:.4f}" if r["similarity"] is not None else "  n/a "
    marker = "  <- author change" if changed else ""
    print(
        f"  [{i:>4}] {r['author_id']:<10} sim={sim_str} new={str(r['is_new_author']):<5}{marker}"
    )

In [ ]:
# Optional: try a quick sweep over a few parameter combinations.
# This is useful when you want to compare settings side-by-side.
for stride_value in [1, 2, 5, 10]:
    for sim_value in [0.90, 0.94, 0.97]:
        text_sample = load_limited_text(DATA_PATH, max_chars=MAX_CHARS, max_sentences=MAX_SENTENCES)
        tracker = AuthorProfileTracker(
            sim_threshold=sim_value,
            ema_alpha=ema_alpha,
            min_tokens_for_update=min_tokens_for_update,
        )
        encoder = DeepStylometryEncoder()
        result = run_pipeline(
            text_sample,
            encoder=encoder,
            tracker=tracker,
            context_window_size=context_window_size,
            stride=stride_value,
            merge_threshold=merge_threshold,
        )
        summary = summarize_run(result)
        print(f"stride={stride_value}, sim_threshold={sim_value}: {summary['num_final_authors']} final authors, {summary['num_sentences']} windows processed")